# Phase 9.1 — Context-Aware WaveToText (Google Colab)

**FLUX Architecture** — Field-based Latent Understanding eXperience

## Overview

Phase 9.1 is a focused enhancement of the **WaveToText decoder** from Phase 9.
Everything else stays frozen — we only retrain the spelling component.

| Metric | Phase 9 | Phase 9.1 Target |
|--------|---------|------------------|
| Chunk accuracy | 82.8% | >93% |
| Word accuracy | 79.8% | >88% |
| Hard spelling | 33% (5/15) | >60% (9/15) |
| UTF-8 multi-byte | 0% | >50% |

### Key Improvements
1. **Left-context awareness**: previous 2 chunks via cross-attention
2. **Larger model**: 512 hidden, 2-layer GRU, 128-dim byte embeddings (~1.6M params)
3. **150K training steps** with cosine LR schedule (6x Phase 9)
4. **UTF-8 augmentation** for non-ASCII characters
5. **Context dropout** regularization (25% per chunk)

### Cell Map

| Cells | Stage |
|-------|-------|
| 1-2 | Clone repo & install deps |
| 3 | Hardware, secrets, logger, config |
| 4 | Smoke test (load Phase 9 checkpoint) |
| 5 | Load data + prepare training pairs |
| 6 | Train ContextWaveToText (150K steps) |
| 7 | Diagnostics |
| 8 | Save checkpoint + upload to HuggingFace |
| 9-11 | Tests 1-3 |
| 12-13 | Demos 1-2 |
| 14 | View results |
| 15 | View full log |
| 16 | Final upload (logs → HF + GitHub) |
| 17 | Save artifacts to Colab output |

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 1 — Clone / Pull Repo
# ═══════════════════════════════════════════════════════════
import os, subprocess, sys

REPO_URL = "https://github.com/Unseengap/FLUX.git"
REPO_DIR = "/content/FLUX"

if os.path.exists(REPO_DIR) and os.path.exists(f"{REPO_DIR}/.git"):
    print("  ℹ Repo exists, pulling latest...")
    subprocess.run(['git', 'pull', '--ff-only'], cwd=REPO_DIR)
else:
    print(f"  ℹ Cloning {REPO_URL}...")
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR])

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(f"  ✓ Working directory: {os.getcwd()}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 2 — Install Dependencies
# ═══════════════════════════════════════════════════════════
!pip install -q -e . 2>/dev/null || pip install -q -r requirements.txt
!pip install -q datasets huggingface_hub rich 2>/dev/null
print("  ✓ Dependencies installed")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 3 — Hardware, Secrets, Logger, Config
# ═══════════════════════════════════════════════════════════
import sys, os, torch
from pathlib import Path

# Path setup
REPO_DIR = '/content/FLUX'
sys.path.insert(0, REPO_DIR)
for _phase in ['phase1', 'phase2', 'phase3', 'phase4', 'phase5',
               'phase6', 'phase7', 'phase8', 'phase9', 'phase9_1']:
    sys.path.insert(0, str(Path(REPO_DIR) / 'phases' / _phase))

from flux_utils import (
    get_device, PhaseLogger, PhaseResults, _resolve_hf_token,
    upload_checkpoint_to_hf, upload_logs_to_hf, git_commit_and_push,
    CHECKPOINT_DIR,
)

# Logger (appends to phase9.log since 9.1 is a sub-phase)
log = PhaseLogger(phase=9)
log.cell_start("Cell 3 — Hardware & Secrets")

# Device
device = get_device()
log.info(f"Device: {device}")
if device == 'cuda':
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    log.info(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")

# HF Token
hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    log.success("HF_TOKEN loaded from Colab secrets")
except Exception:
    hf_token = _resolve_hf_token()
    if hf_token:
        log.success("HF_TOKEN loaded from environment")
    else:
        log.warning("No HF_TOKEN found — uploads will be skipped")

# Config
from train_context_wtt import PHASE9_1_CONFIG
print("\n  Phase 9.1 Config:")
for k, v in PHASE9_1_CONFIG.items():
    print(f"    {k}: {v}")

log.cell_end("Cell 3 — Hardware & Secrets", "PASS")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 4 — Smoke Test (Load Phase 9 Checkpoint)
# ═══════════════════════════════════════════════════════════
log.cell_start("Cell 4 — Smoke Test")

from train_context_wtt import load_phase9_for_9_1, build_context_wtt

# Load frozen CSE + WaveChunker from phase9.phase.pt
cse, chunker, ckpt9 = load_phase9_for_9_1(device=device)

# Quick CSE test
test_wave = cse.encode("hello world")
assert test_wave.full.shape[-1] == 432, f"Expected 432, got {test_wave.full.shape[-1]}"
log.success(f"CSE smoke test: {test_wave.full.shape}")

# Quick chunker test
wave_seq = test_wave.full.to(device)
chunks, spans = chunker(wave_seq)
log.success(f"WaveChunker: {wave_seq.shape[0]} waves → {chunks.shape[0]} chunks")

# Build fresh ContextWaveToText
context_wtt = build_context_wtt(device=device)

# Quick forward test
test_bytes = torch.tensor([104, 101, 108, 108, 111], dtype=torch.long, device=device)
with torch.no_grad():
    logits = context_wtt(chunks[0], test_bytes)
assert logits.shape == (6, 257), f"Expected (6, 257), got {logits.shape}"
log.success(f"ContextWTT forward: logits {logits.shape}")

# Quick decode test
decoded = context_wtt.decode(chunks[0], temperature=0.8)
log.success(f"ContextWTT decode: {len(decoded)} bytes")

log.cell_end("Cell 4 — Smoke Test", "PASS")
print("\n  ✓ All smoke tests passed — ready for training")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 5 — Load Data + Prepare Training Pairs
# ═══════════════════════════════════════════════════════════
log.cell_start("Cell 5 — Data Prep")

from train_context_wtt import (
    load_training_data, Phase9_1_Trainer, UTF8_AUGMENT_TEXTS,
)

# Load training data
texts = load_training_data(max_docs=PHASE9_1_CONFIG['max_train_docs'])
split_idx = int(len(texts) * 0.9)
train_texts = texts[:split_idx]
eval_texts = texts[split_idx:]
print(f"  Train: {len(train_texts):,} docs, Eval: {len(eval_texts):,} docs")

# Create trainer
trainer = Phase9_1_Trainer(
    cse=cse,
    chunker=chunker,
    context_wtt=context_wtt,
    device=device,
    log=log,
)

# Extract training pairs (wave, context, bytes)
training_pairs = trainer.prepare_training_data(
    train_texts,
    max_pairs=500000,
    utf8_texts=UTF8_AUGMENT_TEXTS,
)

log.metric("training_pairs", f"{len(training_pairs):,}")
log.cell_end("Cell 5 — Data Prep", "PASS")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 6 — Train ContextWaveToText (150K steps)
# ═══════════════════════════════════════════════════════════
log.cell_start("Cell 6 — CWTT Train")

result = trainer.train(
    training_pairs,
    max_steps=PHASE9_1_CONFIG['max_steps'],
    batch_size=PHASE9_1_CONFIG['batch_size'],
    grad_accum=PHASE9_1_CONFIG['grad_accum'],
    lr=PHASE9_1_CONFIG['lr'],
    warmup_steps=PHASE9_1_CONFIG['warmup_steps'],
    log_interval=PHASE9_1_CONFIG['log_interval'],
    eval_interval=PHASE9_1_CONFIG['eval_interval'],
    eval_texts=eval_texts,
)

log.cell_end("Cell 6 — CWTT Train", "PASS")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 7 — Diagnostics
# ═══════════════════════════════════════════════════════════
log.cell_start("Cell 7 — Diagnostics")

context_wtt.eval()
import torch

# Diagnostic 1: Chunk-level accuracy sample
print("  📊 Chunk-Level Accuracy (50 texts, max 500 chunks)")
print("  " + "="*56)
chunk_acc = trainer._evaluate_chunk_accuracy(eval_texts[:50], max_chunks=500)
print(f"  Chunk accuracy: {chunk_acc:.1%}")
print(f"  Phase 9 baseline: 82.8%")
print()

# Diagnostic 2: Hard spelling
print("  📊 Hard Spelling Test (15 words in carrier sentences)")
print("  " + "="*56)
from train_context_wtt import HARD_SPELLING_WORDS

correct = 0
for word in HARD_SPELLING_WORDS:
    carrier = f"This is {word} here"
    try:
        wave = cse.encode(carrier)
        wave_seq = wave.full.to(device)
        chunk_waves, spans = chunker(wave_seq)
        decoded_chunks = context_wtt.decode_sequence(chunk_waves, temperature=0.3)
        decoded_text = b''.join(decoded_chunks)
        decoded_str = decoded_text.decode('utf-8', errors='replace')
        found = word in decoded_str
        if found: correct += 1
        status = '✓' if found else '✗'
        print(f"  {status} {word:<20} → '{decoded_str}'")
    except Exception as e:
        print(f"  ✗ {word:<20} → ERROR: {e}")
print(f"\n  Hard spelling: {correct}/{len(HARD_SPELLING_WORDS)} ({correct/len(HARD_SPELLING_WORDS):.0%})")
print(f"  Phase 9 baseline: 5/15 (33%)")
print()

# Diagnostic 3: UTF-8
print("  📊 UTF-8 Multi-Byte Test")
print("  " + "="*56)
utf8_acc = trainer._evaluate_utf8_accuracy()
print(f"  UTF-8 accuracy: {utf8_acc:.1%}")
print(f"  Phase 9 baseline: 0%")
print()

# Diagnostic 4: Context vs no-context comparison
print("  📊 Context Impact Test (5 texts)")
print("  " + "="*56)
ctx_better = 0
ctx_same = 0
ctx_worse = 0
total_chunks_tested = 0

for text in eval_texts[:5]:
    try:
        wave = cse.encode(text)
        wave_seq = wave.full.to(device)
        text_bytes = text.encode('utf-8', errors='replace')
        chunk_waves, spans = chunker(wave_seq)
        for i in range(chunk_waves.shape[0]):
            start, end = spans[i]
            gt = text_bytes[min(start,len(text_bytes)):min(end,len(text_bytes))]
            if len(gt) == 0: continue
            ctx = chunk_waves[max(0,i-2):i] if i > 0 else None
            dec_ctx = context_wtt.decode(chunk_waves[i], 0.3, context_waves=ctx)
            dec_no = context_wtt.decode(chunk_waves[i], 0.3, context_waves=None)
            match_ctx = (dec_ctx == gt)
            match_no = (dec_no == gt)
            if match_ctx and not match_no: ctx_better += 1
            elif match_ctx == match_no: ctx_same += 1
            else: ctx_worse += 1
            total_chunks_tested += 1
    except Exception:
        continue

print(f"  Chunks tested: {total_chunks_tested}")
print(f"  Context helped:  {ctx_better} ({ctx_better/max(total_chunks_tested,1):.0%})")
print(f"  No difference:   {ctx_same} ({ctx_same/max(total_chunks_tested,1):.0%})")
print(f"  Context hurt:    {ctx_worse} ({ctx_worse/max(total_chunks_tested,1):.0%})")

log.metric("diag_chunk_acc", f"{chunk_acc:.1%}")
log.metric("diag_hard_spelling", f"{correct}/{len(HARD_SPELLING_WORDS)}")
log.metric("diag_utf8_acc", f"{utf8_acc:.1%}")
log.metric("diag_ctx_helped", f"{ctx_better}/{total_chunks_tested}")
log.cell_end("Cell 7 — Diagnostics", "PASS")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 8 — Save Checkpoint + Upload to HuggingFace
# ═══════════════════════════════════════════════════════════
log.cell_start("Cell 8 — Save & Upload")

# Save checkpoint
ckpt_path = trainer.save_checkpoint(result, ckpt9)

# Upload to HuggingFace
if hf_token:
    try:
        from huggingface_hub import HfApi
        api = HfApi(token=hf_token)
        try:
            api.create_repo(repo_id='UnseenGAP/FLUX', repo_type='model', exist_ok=True)
        except Exception:
            pass
        api.upload_file(
            path_or_fileobj=str(ckpt_path),
            path_in_repo='checkpoints/phase9_1.phase.pt',
            repo_id='UnseenGAP/FLUX',
            commit_message=f'Phase 9.1 checkpoint — {torch.cuda.get_device_name(0) if device=="cuda" else device}',
        )
        log.success("Phase 9.1 checkpoint uploaded to HuggingFace")
    except Exception as e:
        log.warning(f"HF upload failed: {e}")
else:
    log.warning("No HF token — skipping upload")

log.cell_end("Cell 8 — Save & Upload", "PASS")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 9 — Test 1: Chunk Accuracy
# ═══════════════════════════════════════════════════════════
log.cell_start("Cell 9 — Test 1")

context_wtt.eval()
correct = 0
total = 0

for text in eval_texts[:50]:
    if not text or len(text.strip()) < 10: continue
    try:
        wave = cse.encode(text)
        wave_seq = wave.full.to(device)
        text_bytes = text.encode('utf-8', errors='replace')
        chunk_waves, spans = chunker(wave_seq)
        for i in range(chunk_waves.shape[0]):
            start, end = spans[i]
            gt = text_bytes[min(start,len(text_bytes)):min(end,len(text_bytes))]
            if len(gt) == 0: continue
            ctx = chunk_waves[max(0,i-2):i] if i > 0 else None
            decoded = context_wtt.decode(chunk_waves[i], 0.5, context_waves=ctx)
            if decoded == gt: correct += 1
            total += 1
            if total >= 500: break
    except: continue
    if total >= 500: break

chunk_accuracy = correct / max(total, 1)
passed = chunk_accuracy >= 0.93
print(f"  Chunk accuracy: {correct}/{total} ({chunk_accuracy:.1%})")
print(f"  Threshold: ≥93%  |  {'✓ PASS' if passed else '✗ FAIL'}")
log.metric("test1_chunk_acc", f"{chunk_accuracy:.1%}")
log.cell_end("Cell 9 — Test 1", "PASS" if passed else "FAIL")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 10 — Test 2: Hard Word Spelling
# ═══════════════════════════════════════════════════════════
log.cell_start("Cell 10 — Test 2")

context_wtt.eval()
correct = 0
total = len(HARD_SPELLING_WORDS)

for word in HARD_SPELLING_WORDS:
    carrier = f"This is {word} here"
    try:
        wave = cse.encode(carrier)
        wave_seq = wave.full.to(device)
        chunk_waves, spans = chunker(wave_seq)
        decoded_chunks = context_wtt.decode_sequence(chunk_waves, temperature=0.3)
        decoded_text = b''.join(decoded_chunks)
        decoded_str = decoded_text.decode('utf-8', errors='replace')
        if word in decoded_str: correct += 1
        status = '✓' if word in decoded_str else '✗'
        print(f"  {status} {word:<20} → '{decoded_str}'")
    except Exception as e:
        print(f"  ✗ {word:<20} → ERROR")

hard_acc = correct / max(total, 1)
passed = hard_acc >= 0.60
print(f"\n  Hard spelling: {correct}/{total} ({hard_acc:.0%})")
print(f"  Threshold: ≥60%  |  {'✓ PASS' if passed else '✗ FAIL'}")
log.metric("test2_hard_spelling", f"{hard_acc:.0%}")
log.cell_end("Cell 10 — Test 2", "PASS" if passed else "FAIL")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 11 — Test 3: UTF-8 Multi-Byte
# ═══════════════════════════════════════════════════════════
log.cell_start("Cell 11 — Test 3")

context_wtt.eval()
utf8_words = ["café", "naïve", "résumé", "piñata", "Zürich",
              "château", "crème", "über", "señor", "François"]
correct = 0

for word in utf8_words:
    carrier = f"The word is {word} exactly"
    try:
        wave = cse.encode(carrier)
        wave_seq = wave.full.to(device)
        chunk_waves, spans = chunker(wave_seq)
        decoded_chunks = context_wtt.decode_sequence(chunk_waves, temperature=0.3)
        decoded_text = b''.join(decoded_chunks)
        decoded_str = decoded_text.decode('utf-8', errors='replace')
        if word in decoded_str: correct += 1
        status = '✓' if word in decoded_str else '✗'
        print(f"  {status} {word:<15} → '{decoded_str}'")
    except Exception as e:
        print(f"  ✗ {word:<15} → ERROR")

utf8_acc = correct / len(utf8_words)
passed = utf8_acc >= 0.50
print(f"\n  UTF-8 accuracy: {correct}/{len(utf8_words)} ({utf8_acc:.0%})")
print(f"  Threshold: ≥50%  |  {'✓ PASS' if passed else '✗ FAIL'}")
log.metric("test3_utf8_acc", f"{utf8_acc:.0%}")
log.cell_end("Cell 11 — Test 3", "PASS" if passed else "FAIL")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 12 — Demo 1: Spelling Comparison
# ═══════════════════════════════════════════════════════════
log.cell_start("Cell 12 — Demo 1")

context_wtt.eval()
demo_phrases = [
    "The quick brown fox jumps over the lazy dog.",
    "Energy equals mass times the speed of light squared.",
    "Quantum mechanics describes the behavior of particles.",
    "The fundamental relationship between science and technology.",
    "Neural networks learn from data through backpropagation.",
    "The café serves crème brûlée and espresso every morning.",
]

print("  Spelling Demo — Phase 9.1 ContextWaveToText")
print("  " + "="*56)

for phrase in demo_phrases:
    try:
        wave = cse.encode(phrase)
        wave_seq = wave.full.to(device)
        chunk_waves, spans = chunker(wave_seq)
        decoded_chunks = context_wtt.decode_sequence(chunk_waves, temperature=0.3)
        decoded_text = b''.join(decoded_chunks)
        decoded_str = decoded_text.decode('utf-8', errors='replace')
        match = phrase == decoded_str
        status = '✓' if match else '~'
        print(f"\n  {status} Original: \"{phrase}\"")
        print(f"    Decoded:  \"{decoded_str}\"")
    except Exception as e:
        print(f"\n  ✗ \"{phrase[:40]}...\" → ERROR: {e}")

log.cell_end("Cell 12 — Demo 1", "PASS")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 13 — Demo 2: Context Impact Visualization
# ═══════════════════════════════════════════════════════════
log.cell_start("Cell 13 — Demo 2")

context_wtt.eval()
viz_text = "Scientists have discovered a fundamental relationship in quantum physics."

print(f"  Input: \"{viz_text}\"")
print(f"  {'─'*60}")

wave = cse.encode(viz_text)
wave_seq = wave.full.to(device)
text_bytes = viz_text.encode('utf-8', errors='replace')
chunk_waves, spans = chunker(wave_seq)

print(f"  {wave_seq.shape[0]} byte-waves → {chunk_waves.shape[0]} chunks\n")

for i in range(chunk_waves.shape[0]):
    start, end = spans[i]
    gt = text_bytes[min(start,len(text_bytes)):min(end,len(text_bytes))]
    gt_str = gt.decode('utf-8', errors='replace')
    
    ctx = chunk_waves[max(0,i-2):i] if i > 0 else None
    dec_ctx = context_wtt.decode(chunk_waves[i], 0.3, context_waves=ctx)
    dec_no  = context_wtt.decode(chunk_waves[i], 0.3, context_waves=None)
    
    str_ctx = dec_ctx.decode('utf-8', errors='replace')
    str_no  = dec_no.decode('utf-8', errors='replace')
    
    m_ctx = '✓' if dec_ctx == gt else '✗'
    m_no  = '✓' if dec_no == gt else '✗'
    star = ' ★' if (dec_ctx == gt and dec_no != gt) else ''
    
    print(f"  [{i:>2}] GT='{gt_str:<12}' {m_ctx} ctx='{str_ctx:<12}' {m_no} no='{str_no:<12}'{star}")

print(f"\n  ★ = context fixed a decode error")
log.cell_end("Cell 13 — Demo 2", "PASS")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 14 — Generate Results
# ═══════════════════════════════════════════════════════════
log.cell_start("Cell 14 — Results")

results = PhaseResults(phase=9, component_name="Context-Aware WaveToText (Phase 9.1)")
results.add_test(
    "Chunk Accuracy", result.chunk_accuracy >= 0.93,
    score=f"{result.chunk_accuracy:.1%}", threshold="≥93%",
)
results.add_test(
    "Word Accuracy", result.word_accuracy >= 0.88,
    score=f"{result.word_accuracy:.1%}", threshold="≥88%",
)
results.add_test(
    "Hard Spelling", result.hard_spelling_accuracy >= 0.60,
    score=f"{result.hard_spelling_accuracy:.1%}", threshold="≥60%",
)
results.add_test(
    "UTF-8 Accuracy", result.utf8_accuracy >= 0.50,
    score=f"{result.utf8_accuracy:.1%}", threshold="≥50%",
)
results.add_metric("Training steps", result.total_steps)
results.add_metric("Final loss", f"{result.final_loss:.4f}")
results.add_metric("Average loss", f"{result.avg_loss:.4f}")
results.add_metric("Training time", f"{result.total_time_seconds:.1f}s ({result.total_time_seconds/60:.1f} min)")
results.add_metric("Parameters", f"{sum(p.numel() for p in context_wtt.parameters()):,}")

results_path = str(Path(REPO_DIR) / 'phases' / 'phase9_1' / 'RESULTS_PHASE_9_1.md')
results.save(path=results_path)

log.cell_end("Cell 14 — Results", "PASS")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 15 — View Full Log
# ═══════════════════════════════════════════════════════════
print(log.get_log_contents()[-5000:])

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 16 — Final Upload (Logs → HF + GitHub)
# ═══════════════════════════════════════════════════════════
log.cell_start("Cell 16 — Final Upload")

if hf_token:
    upload_logs_to_hf(phase=9, hf_token=hf_token)
    # Also upload results
    try:
        from huggingface_hub import HfApi
        api = HfApi(token=hf_token)
        results_file = Path(REPO_DIR) / 'phases' / 'phase9_1' / 'RESULTS_PHASE_9_1.md'
        if results_file.exists():
            api.upload_file(
                path_or_fileobj=str(results_file),
                path_in_repo='results/RESULTS_PHASE_9_1.md',
                repo_id='UnseenGAP/FLUX',
                commit_message='Phase 9.1 results',
            )
            log.success("Results uploaded to HuggingFace")
    except Exception as e:
        log.warning(f"Results upload failed: {e}")

# Git commit
git_commit_and_push(
    files=['logs/phase9.log', 'phases/phase9_1/RESULTS_PHASE_9_1.md'],
    message='Phase 9.1 — Context-Aware WaveToText results',
    repo_dir=REPO_DIR,
)

log.cell_end("Cell 16 — Final Upload", "PASS")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 17 — Save Artifacts to Colab Output
# ═══════════════════════════════════════════════════════════
import shutil
from pathlib import Path

output_dir = Path('/content/output')
output_dir.mkdir(exist_ok=True)

# Copy checkpoint
ckpt_src = Path(REPO_DIR) / 'checkpoints' / 'phase9_1.phase.pt'
if ckpt_src.exists():
    shutil.copy2(str(ckpt_src), str(output_dir / 'phase9_1.phase.pt'))
    print(f"  ✓ Checkpoint copied to {output_dir / 'phase9_1.phase.pt'}")

# Copy log
log_src = Path(REPO_DIR) / 'logs' / 'phase9.log'
if log_src.exists():
    shutil.copy2(str(log_src), str(output_dir / 'phase9.log'))
    print(f"  ✓ Log copied")

# Copy results
res_src = Path(REPO_DIR) / 'phases' / 'phase9_1' / 'RESULTS_PHASE_9_1.md'
if res_src.exists():
    shutil.copy2(str(res_src), str(output_dir / 'RESULTS_PHASE_9_1.md'))
    print(f"  ✓ Results copied")

print(f"\n  ✓ All artifacts saved to {output_dir}")
print(f"\n{'='*60}")
print(f"  Phase 9.1 Complete!")
print(f"{'='*60}")